# 02 — Seed LearnSphere with deterministic fake data

Generates a small, self-consistent dataset across the five source tables. The
seed uses a fixed random seed so re-runs are reproducible — handy when
diffing sync output during development.

Volumes by default:

| Table | Rows |
| --- | --- |
| `instructors` | 25 |
| `courses` | 100 |
| `learners` | 500 |
| `enrollments` | ~2 000 |
| `course_reviews` | ~800 |

Tweak the constants below to scale up.


In [ ]:
# Load .env from the repo root so we use the same settings as the sync job.
from pathlib import Path
import os

from dotenv import load_dotenv

repo_root = Path.cwd()
while not (repo_root / ".env.example").exists():
    if repo_root == repo_root.parent:
        raise RuntimeError("could not find repo root (.env.example missing)")
    repo_root = repo_root.parent

load_dotenv(repo_root / ".env")

BQ_PROJECT = os.environ["BQ_PROJECT_ID"]
BQ_DATASET = os.environ.get("BQ_DATASET", "learnsphere")
BQ_LOCATION = os.environ.get("BQ_LOCATION", "US")
print(f"project={BQ_PROJECT}  dataset={BQ_DATASET}  location={BQ_LOCATION}")


In [ ]:
import random
import uuid
from datetime import datetime, timedelta, timezone

from faker import Faker

SEED = 42
random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

N_INSTRUCTORS = 25
N_COURSES = 100
N_LEARNERS = 500
N_ENROLLMENTS = 2_000
N_REVIEWS = 800

NOW = datetime.now(timezone.utc).replace(microsecond=0)
CATEGORIES = ["tech", "design", "business", "language", "wellness"]
LEVELS = ["beginner", "intermediate", "advanced"]
PLANS = ["free", "pro", "team"]
STATUSES = ["active", "completed", "dropped"]


In [ ]:
def _ts_within(days: int) -> datetime:
    delta = timedelta(seconds=random.randint(0, days * 86_400))
    return NOW - delta

def _iso(dt: datetime) -> str:
    return dt.isoformat()

instructors = [
    {
        "instructor_id": f"I-{i:04d}",
        "display_name": fake.name(),
        "bio": fake.sentence(nb_words=12),
        "country": fake.country_code(),
        "rating": round(random.uniform(3.5, 5.0), 2),
        "updated_at": _iso(_ts_within(365)),
    }
    for i in range(N_INSTRUCTORS)
]

courses = []
for i in range(N_COURSES):
    courses.append({
        "course_id": f"C-{i:05d}",
        "title": fake.catch_phrase(),
        "category": random.choice(CATEGORIES),
        "level": random.choice(LEVELS),
        "duration_minutes": random.choice([30, 45, 60, 90, 120, 180]),
        "price_usd": round(random.choice([0, 9.99, 19.99, 29.99, 49.99]), 2),
        "instructor_id": random.choice(instructors)["instructor_id"],
        "updated_at": _iso(_ts_within(180)),
    })

learners = [
    {
        "learner_id": f"L-{i:05d}",
        "email": f"learner{i}@example.com",
        "display_name": fake.name(),
        "country": fake.country_code(),
        "signup_date": _ts_within(720).date().isoformat(),
        "plan_tier": random.choice(PLANS),
        "updated_at": _iso(_ts_within(60)),
    }
    for i in range(N_LEARNERS)
]
print(f"in-memory: {len(instructors)} instructors  {len(courses)} courses  {len(learners)} learners")


In [ ]:
enrollments = []
enrollment_keys: set[tuple[str, str]] = set()
while len(enrollments) < N_ENROLLMENTS:
    learner = random.choice(learners)
    course = random.choice(courses)
    key = (learner["learner_id"], course["course_id"])
    if key in enrollment_keys:
        continue
    enrollment_keys.add(key)
    enrolled = _ts_within(540)
    last_activity = enrolled + timedelta(days=random.randint(0, 90))
    status = random.choices(STATUSES, weights=[6, 3, 1])[0]
    progress = 100.0 if status == "completed" else round(random.uniform(0, 99), 1)
    enrollments.append({
        "enrollment_id": str(uuid.UUID(int=random.getrandbits(128))),
        "learner_id": learner["learner_id"],
        "course_id": course["course_id"],
        "enrolled_at": _iso(enrolled),
        "progress_percent": progress,
        "status": status,
        "last_activity_at": _iso(min(last_activity, NOW)),
    })

reviews = []
for _ in range(N_REVIEWS):
    enr = random.choice(enrollments)
    reviews.append({
        "review_id": str(uuid.UUID(int=random.getrandbits(128))),
        "course_id": enr["course_id"],
        "learner_id": enr["learner_id"],
        "rating": random.randint(1, 5),
        "comment": fake.sentence(nb_words=10),
        "created_at": _iso(_ts_within(180)),
    })

print(f"in-memory: {len(enrollments)} enrollments  {len(reviews)} reviews")


## Load into BigQuery


In [ ]:
import io
import json

from google.cloud import bigquery

bq = bigquery.Client(project=BQ_PROJECT, location=BQ_LOCATION)

def _load(table: str, rows: list[dict]) -> None:
    table_ref = f"{BQ_PROJECT}.{BQ_DATASET}.{table}"
    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
    )
    payload = io.BytesIO("\n".join(json.dumps(r) for r in rows).encode("utf-8"))
    job = bq.load_table_from_file(payload, table_ref, job_config=job_config)
    job.result()
    print(f"  loaded {len(rows):>5} into {table}")

_load("instructors", instructors)
_load("learners", learners)
_load("courses", courses)
_load("enrollments", enrollments)
_load("course_reviews", reviews)
print("done")


Next: `03_validate.ipynb` runs the exact JOINs the sync uses, locally, so you
can confirm the dataset is well-shaped before pointing the sync job at it.
